## Advanced Python: Applied Machine Learning Techniques
### Day 4: Machine Learning using `PyTorch`
##### Planned Topics
  - PyTorch Features (a quick recap)
    - Configuring GPU acceleration in PyTorch tensors
    - Converting between `numpy` arrays to PyTorch `tensor`
  - An overview on automatic differentiation
  - Training and Prediction using Linear Regression
  - The PyTorch training loop: workflow
  - An overview on Machine-Learning Taxonomy
  - An overview on Artificial Neural Networks
  - Building a simple image classifier model using ```PyTorch```


---

###  `PyTorch` concepts recap 

### Getting started with PyTorch
#### Setting up PyTorch environment in Anaconda:
```
  conda create -n pytorch_exploration pytorch torchvision torchaudio numpy sklearn matplotlib

  # For nvidia GPUs
  conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia
```

In [1]:
import torch
torch.__version__

'2.9.1'

In [2]:
print(torch.__config__.show())

PyTorch built with:
  - GCC 4.2
  - C++ Version: 201703
  - clang 20.1.8
  - OpenMP 202011
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: DEFAULT
  - Build settings: BLAS_INFO=open, BUILD_TYPE=Release, COMMIT_SHA=Unknown, CXX_COMPILER=/Users/ec2-user/croot/libtorch_1774862146849/_build_env/bin/arm64-apple-darwin20.0.0-clang++, CXX_FLAGS=-ftree-vectorize -fPIC -fstack-protector-strong -O2 -pipe -stdlib=libc++  -fmessage-length=0 -isystem /Users/ec2-user/croot/libtorch_1774862146849/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_pla/include -fdebug-prefix-map=/Users/ec2-user/croot/libtorch_1774862146849/work=/usr/local/src/conda/libtorch-2.9.1 -fdebug-prefix-map=/Users/ec2-user/croot/libtorch_1774862146849/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_

In [3]:
torch.accelerator.is_available()

True

In [4]:
torch.cuda.is_available()

False

In [5]:
torch.mps.is_available()

True

In [6]:
device = torch.accelerator.current_accelerator()
print(repr(device))
print(device.type)

device(type='mps')
mps


In [7]:
device = torch.accelerator.current_accelerator().type \
    if torch.accelerator.is_available() else "cpu"
print(device)

mps


In [8]:
print("Number of devices:", torch.accelerator.device_count())
print("Current device index:", torch.accelerator.current_device_index())
print("Current device:", torch.accelerator.current_accelerator())

Number of devices: 1
Current device index: 0
Current device: mps


In [9]:
torch.mps.current_allocated_memory()

0

In [11]:
torch.mps.recommended_max_memory() / ( 1024 * 1024 * 1024) # In GiB

17.760009765625

#### PyTorch terminologies
 - Scalars -> int / float / bool (0-d data-structure)
 - Vectors -> list / np.array (1-d arrays)
 - Matrics -> 2-d arrays
 - Tensors -> n-d arrays

Tensor is a common term used for 0-d, 1-d, 2-d, n-d arrays  


In [12]:
t = torch.tensor(10) # A tensor representing scalar value
print(t, t.ndim, t.shape, t.dtype)
print(t.size(), t.item(), t.numel())

tensor(10) 0 torch.Size([]) torch.int64
torch.Size([]) 10 1


In [13]:
# A tensor representing a vector
# 1-D tensor
# Elements of a tensor can be accessed using indexing and slicing
t = torch.tensor([2, 4, 1, 7, 8])
print(t, type(t))
print(t.ndim, t.shape, t.numel())
print(t[0], t[-1], t[:3])
print(type(t[0]))

tensor([2, 4, 1, 7, 8]) <class 'torch.Tensor'>
1 torch.Size([5]) 5
tensor(2) tensor(8) tensor([2, 4, 1])
<class 'torch.Tensor'>


In [14]:
import numpy as np
a = np.array([2, 4, 1, 7, 8])
print(a)
print(a[0], type(a[0]))

[2 4 1 7 8]
2 <class 'numpy.int64'>


In [15]:
t = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(t)

tensor([[1, 2, 3],
        [4, 5, 6]])


In [16]:
data = [[11, 22, 33], [44, 55, 66], [77, 88, 99]]
print(data, type(data))

a = np.array(data)
print(a, type(a), a.size)

t = torch.tensor(data)
print(t, type(t), t.size())

[[11, 22, 33], [44, 55, 66], [77, 88, 99]] <class 'list'>
[[11 22 33]
 [44 55 66]
 [77 88 99]] <class 'numpy.ndarray'> 9
tensor([[11, 22, 33],
        [44, 55, 66],
        [77, 88, 99]]) <class 'torch.Tensor'> torch.Size([3, 3])


In [17]:
print(a.size)
print(t.size(), t.numel())

print(a.shape)
print(t.shape)

9
torch.Size([3, 3]) 9
(3, 3)
torch.Size([3, 3])


In [18]:
a = np.arange(12).reshape(3, 4)
a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

In [19]:
t = torch.arange(12).reshape(3, 4) # It can make a copy of the underlying data under certain circumstances.
t

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [20]:
t = torch.arange(12).view(3, 4)
t

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

#### Reshape vs View

In [ ]:

t = torch.arange(24).reshape(6, 4)
t1 = t.T
print(t1, t1.is_contiguous())


tensor([[ 0,  4,  8, 12, 16, 20],
        [ 1,  5,  9, 13, 17, 21],
        [ 2,  6, 10, 14, 18, 22],
        [ 3,  7, 11, 15, 19, 23]]) False


In [27]:
t2 = t1.reshape(6, 4)
print(t2, t2.is_contiguous())

tensor([[ 0,  4,  8, 12],
        [16, 20,  1,  5],
        [ 9, 13, 17, 21],
        [ 2,  6, 10, 14],
        [18, 22,  3,  7],
        [11, 15, 19, 23]]) True


In [28]:
t3 = t1.view(6, 4)
print(t3, t3.is_contiguous())

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [ ]:
data = [[11, 22, 33], [44, 55.5, 66], [77, 88, 99]]

t = torch.tensor(data)
t

In [ ]:
data = [[11, 22, 33], [44, 55.5, 66], [77, 88, 99]]

t = torch.tensor(data, dtype=torch.int)
t

#### PyTorch numerical data-types

In [ ]:
print(torch.int, torch.int32)
print(torch.short, torch.int16)
print(torch.int8)  # torch.byte
print(torch.float, torch.float32)
print(torch.double, torch.float64)
print(torch.half, torch.float16)
print(torch.bool)

#### Creating tensors from `numpy` array

In [ ]:
# Creating a tensor from a numpy array (simpler way that creates a copy!)
a = np.array([10, 20, 30, 40])
print(a, type(a))

t = torch.tensor(a) # Always creates a new tensor by copying the contents
print(t, type(t))

a[0] = 12
print(a, t)

In [ ]:
# Creating a tensor from a numpy array (optimal way - shares data!)
a = np.array([10, 20, 30, 40])
print(a, type(a))

t = torch.from_numpy(a) # Creates a tensor by sharing the contents from numpy array
print(t, type(t))

a[0] = 12
print(a, t)

t[1] = 25
print(a, t)

In [ ]:
# Converting tensor to numpy arrays

t = torch.tensor([11, 22, 33, 44])

a = t.numpy() # Creates a new numpy array from a tensor while sharing data
print(a)

t[0] = 10
print(t, a)

In [ ]:
t = torch.tensor([11, 22, 33, 44], device="mps:0") # device="mps:0" for Mac, device="cuda:0" for Windows/Linux with NVIDIA GPU
print(t)
t += 1
print(t)

In [ ]:
torch.mps.current_allocated_memory()

In [ ]:
print(t.device)
print(t.device.type, t.device.index)

In [ ]:
t1 = torch.tensor([1, 2, 3, 4, 5])
t1.device.index

In [ ]:
t = torch.arange(1, 10_000, dtype=torch.int64, device="mps")
t.device

t1 = torch.tensor(t, device="mps")
t1.device

In [ ]:
t1 = torch.arange(10000)
print(t1.device)

t2 = torch.arange(10000, device="mps")
print(t2.device)


In [ ]:
t = torch.arange(1_000_000)
%timeit z = t + 2

In [ ]:
t = torch.arange(1_000_000, device="mps")
%timeit z =  t + 2

In [ ]:
print(x)
print(x.dtype)
print(x.device)
print(x.shape)
print(x.size())
print(x.numel()) # Equivalent to np.array.size()
print(x.requires_grad)


#### Scalars, Vectors, Matrices and ND-Arrays as ```tensors```

In [ ]:
s = torch.tensor(5)                      # Scalar (0-d)
v = torch.tensor([2, 5, 7, 9])           # Vector (1-d)
m = torch.tensor([[7, 2, 1], [8, 4, 3]]) # Matrix (2-d array)
t = torch.tensor([[                      # ND-Array 
                   [1, 2], [3, 4],
                   [4, 5], [6, 7]
                   ],
                   [
                    [2, 1], [9, 3],
                    [3, 8], [4, 6]
                   ]])

print(s.shape, v.shape, m.shape, t.shape)
print(s.ndim, v.ndim, m.ndim, t.ndim)


##### ```PyTorch``` datatypes

In [ ]:
# tensor converting to compatible data-types
tensor = torch.tensor(1)
print(tensor.bool())
print(tensor.short())  # int16
print(tensor.long())
print(tensor.half())  # float16
print(tensor.float()) # float32
print(tensor.double()) # float64

In [ ]:
a = torch.tensor(1, dtype=torch.int8)
print(a, a.dtype)

In [ ]:
a = torch.tensor(1)
print(a, a.dtype)
b = a.type(torch.int32)
print(a.dtype, b.dtype)

In [ ]:
a = torch.tensor(1)
print(a, a.dtype)
b = a.type(torch.int8)
print(a.dtype, b.dtype)

In [ ]:
t1 = torch.tensor(1) # Default datatype is either int64 or float32
t2 = torch.tensor(1.0)
print(t1.dtype, t2.dtype)

t1 = torch.tensor(1, dtype=torch.int8)
print(t1, t1.dtype)

t2 = torch.tensor(1, dtype=torch.float16)
print(t2, t2.dtype)

In [ ]:
a = torch.tensor([11, 22, 33])
print(a, a.dtype)

b = a.type(torch.int8)
print(b, b.dtype)
b[0] = 15

print(a, b)

##### Convert between ```numpy arrays``` and ```PyTorch tensors```

In [ ]:
# Convert from numpy array to torch.tensors
import numpy as np
import torch
a = np.array([10, 20, 30])
print(a, type(a))

b = torch.from_numpy(a)
print(b, type(b))

a[0] = 11
b[1] = 22
print(a, b) # When using torch.from_numpy(), 
            # the data between np.array and torch.tensor are shared


In [ ]:
c = torch.tensor([10, 20, 30])
print(c, type(c))

d = c.numpy()
print(d, type(d))

c[0] = 11
d[1] = 22
print(c, d) # Creating numpy array using .numpy() also shares data

In [ ]:
a = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(a.shape, a.size())
print(a.ndim, a.ndimension())
print(a.itemsize)
print(a.numel()) # This is equivalent to np.array.size()

#### Reshaping tensors

In [ ]:
a = torch.arange(9)
print(a)
b = a.reshape(3, 3) # Can work on contiguous / noncontiguous memory, might create a copy
print(b)
c = a.view(3, 3) # Works only on contiguous memory, does not create a copy
print(c)

In [ ]:
a = torch.arange(12).reshape(4, 3)
print(a, a.shape)
b = a.T
print(b, b.shape)

##### Squeeze and Unsqueeze operations

In [ ]:
a = torch.tensor([[[10, 20, 30]]])
print(a, a.shape, a[0, 0, 0])
b = a.squeeze()
print(b, b.shape, b[0])

In [ ]:
a = torch.tensor([10, 20, 30])
print(a, a.shape)
b = a.unsqueeze(dim=1)
print(b, b.shape)

In [ ]:
a = torch.arange(12).reshape(4, 3)
print(a, a.shape)

b = a.unsqueeze(dim=1)
print(b, b.shape)

In [ ]:
a = torch.tensor([10, 20, 30, 40])
a.unsqueeze(dim=1)

In [ ]:
a = torch.tensor([10, 20, 30, 40, 50])
a.view(-1, 1)

#### PyTorch array creation functions

In [ ]:
t = torch.tensor([11, 22, 33])
t

In [ ]:
t = torch.arange(10, 20, 0.5)
t

In [ ]:
t = torch.ones(3, 3)
t

In [ ]:
torch.zeros(2, 2)

In [ ]:
torch.empty(3, 3)

In [ ]:
torch.rand(3, 3)

In [ ]:
torch.randint(10, 20, size=(3, 3))

In [ ]:
torch.linspace(10, 20, 5)

#### ```tensor``` math

In [ ]:
x = torch.tensor([10, 20, 30])
y = torch.tensor([40, 50, 60])
print(x + y)
print(x - y)
print(x * y)
print(x / y)
print(x // y)
print(x % y)

In [ ]:
x = torch.arange(9).reshape(3, 3)
y  = torch.arange(1, 4).unsqueeze(dim=1)
print(x, y, sep="\n")

y1 = torch.arange(1, 4)
print(y1)

z = x @ y # x.dot(y), torch.dot(x, y)
print(z)

print(x @ y1)

In [ ]:
# Element-wise (broadcast operations)
print(x)
print(x + 1)
print(x * 2)
print(x ** 2)

In [ ]:
a = torch.tensor([2, 5, 3])
b = torch.tensor([4, 5, 6]).reshape(3, 1)
print(a)
print(b)
a + b

In [ ]:
z = torch.add(x, y)
z

In [ ]:
a = torch.tensor([10, 20])
print(a.add(5))
print(a)

In [ ]:
a = torch.tensor([10, 20])
print(a.add_(5))
print(a)

b = a.add_(5)
print(a, b)

In [ ]:
a = torch.tensor([10, 20])
a += 5
print(a)

In [ ]:
a = torch.tensor([10, 20])
b = a += 5 # This wont work - Python syntax does not allow this

In [ ]:
y.add_(x)  # all methods ending with _ performs in-place operation
y

In [ ]:
z = x - y
print(z)
z = torch.sub(x, y)
print(z)

In [ ]:
z = torch.mul(x, y)
print(z)

In [ ]:
x = torch.arange(12).reshape(3, 4)
x[1, 1]

In [ ]:
x[1, 1].item()

In [ ]:
y = x.view(12)
y

In [ ]:
x = torch.rand(4, 4)
print(x.shape)
y = x.view(-1, 8)
print(y.shape)

##### Moving Tensors between CPU and GPU

In [ ]:
dev = torch.device("mps") if torch.mps.is_available() else torch.device("cpu")
dev

In [ ]:
a = torch.tensor(3.0, device=dev)
print(a)
print(a.device, a.device.type, a.device.index)

In [ ]:
b = a.to("cpu")
print(a)
print(b)

c = a.cpu()
print(c)

d = c.to(dev)
print(d)

In [ ]:
a = torch.arange(10, device=dev)
print(a)

a.numpy()

In [ ]:
a = torch.arange(10, device=dev)
print(a)

a.cpu().numpy()

In [ ]:
a = np.arange(10)
print(a, type(a))
b = torch.from_numpy(a)
print(b)
c = b.to(dev)
print(a, b, c, sep="\n")


In [ ]:
x = torch.arange(10, device="cpu")
y = torch.arange(10, device="mps")
%timeit x + 1
%timeit y + 1
# NOTE: GPU operations can incur initiation latencies 
# (due to I/O bus delays for programming the GPU to perform an operation)

In [ ]:
x = torch.arange(10_000_000, device="cpu")
y = torch.arange(10_000_000, device="mps")
%timeit x + 1
%timeit y + 1
# GPUs are optimal for large vectorization operations
# Here, it is fast because the number of times GPU instructions
# issued by the CPU is far lesser (reducing I/O bus delays from CPU to GPU).

#### Experimenting with autograd feature
 - Automatic Gradient Computation / Auto-differentiation

In [ ]:
x = torch.arange(4.0, requires_grad=True)
print(x)

In [ ]:
x = torch.tensor(6.0, requires_grad=True)
print(f"{x=}")

y = x ** 2
print(f"{y=}")

print(f"{x=}, {x.grad=}")

y.backward() # dy/dx

print(f"{x.grad=}")

x.grad == 2*x


In [ ]:
# Gradients accumulate over multiple operations when enabled.
x = torch.tensor(3.0, requires_grad=True)
print(f"{x=}, {x.grad=}")

y = x + 2
print(f"{y=}")

z = y * 2
print(f"{z=}")

z.backward()  # dz/dx
print(f"{x.grad=}")

y.backward() # dy/dx
print(f"{x.grad=}")

In [ ]:
# Gradients computations can be disabled by detach operation.
x = torch.tensor(3.0, requires_grad=True)
print(f"{x=}, {x.grad=}")

y = x.detach() + 2
print(f"{y=}")

z = y * 2
print(f"{z=}")

z.backward()  # This operation will fails as gradient computation is detached.
print(f"{x.grad=}")

y.backward() # dy/dx
print(f"{x.grad=}")

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = x + 2
print(f"{y=}")

with torch.no_grad():
    y1 = x + 4
    print(f"{y1=}")

y.backward()
print(x.grad)


In [ ]:
x = torch.tensor(3.0, requires_grad=True)

y = x + 2
print(f"{y=}")

x.requires_grad_(False)
y1 = x + 4
print(f"{y1=}")

x.requires_grad_(True)

y.backward()
print(x.grad)



In [ ]:
a = torch.tensor(6.0)
print(a)
a.requires_grad_(True)
print(a)

In [ ]:
# Another example of gradient accumulation (not desired in this scenario)
weights = torch.ones(4, requires_grad=True)

for epoch in range(5):
    model_output = (weights * 3).sum()

    model_output.backward()
    print(f"{weights=}")
    print(f"{weights.grad=}")
    print("-" * 40)
    

In [ ]:
# Another example of gradient accumulation (not desired in this scenario)
weights = torch.ones(4, requires_grad=True)

for epoch in range(5):
    model_output = (weights * 3).sum()

    model_output.backward()
    print(f"{weights=}")
    print(f"{weights.grad=}")
    print("-" * 40)
    weights.grad.zero_()
    

### Building and Training a Model in PyTorch

In [ ]:
# y = mx + b  -> y = weight * X + bias  # X -> represents a feature

In the example below, we have pre-determined (preset) the weight and the bias, because we wanted to generate 
training data based on "known" values.

But the real scenario for machine learning involves -> determining the weight and the bias by the training model using training data

In [ ]:
weight = 0.6
bias = 0.4

X = torch.arange(0, 1000, 0.1).unsqueeze(dim=1)
print(X[:10])

y = weight * X + bias # y = mx + b # The 'weight' and 'bias' are also known as 'parameters'
print(y[:10])
print(len(X), len(y))

In [ ]:
# Split the known data (training data) -> train:test split 
train_split = int(0.8 * len(X))
print(train_split)

X_train = X[:train_split]
y_train = y[:train_split]

X_test = X[train_split:]
y_test = y[train_split:]

print(len(X_train), len(y_train), len(X_test), len(y_test))
print(X_train[:5], y_train[:5], sep="\n")

In [ ]:
from torch.utils.data import random_split
a = list(range(20))
train_data, test_data = random_split(a, [15, 5])
print(train_data, test_data)
print(list(train_data), list(test_data))

In [ ]:
# conda install scikit-learn matplotlib pandas

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)
print(len(X_train), len(X_test), len(y_train), len(y_test))
print(X_train[:5], y_train[:5])

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def plot_predictions(train_data=X_train, 
                     train_labels=y_train,
                     test_data=X_test,
                     test_labels=y_test,
                     predictions=None):
    
    # Plot training data in blue
    plt.figure(figsize=(10, 7))
    plt.scatter(train_data, train_labels, c="b", s=4, label="Training data")

    # Plot test data in green
    plt.scatter(test_data, test_labels, c="y", s=4, label="Test data")
   
    if predictions is not None:
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")

    plt.legend(prop={"size": 14})

plot_predictions()
plt.show()


In [ ]:
torch.randn(1, requires_grad=True)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(1, requires_grad=True))
        self.bias = nn.Parameter(torch.randn(1, requires_grad=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weight * x + self.bias                                     

In [ ]:
model = LinearRegressionModel()
model

In [ ]:
list(model.parameters())

In [ ]:
model.state_dict()

Predict the outcome without any training

In [ ]:
# Make predictions (for model evaluations)
#with torch.no_grad():
with torch.inference_mode():
    y_preds = model(X_test)

y_preds[:5], y_test[:5]

In [ ]:
plot_predictions(predictions=y_preds)
plt.plot()

In [ ]:
model = LinearRegressionModel()

loss_fn = nn.L1Loss()  # Mean Absolute Error (MAE)

optimizer = optim.SGD(params=model.parameters(), lr=0.01)  # lr -> Learning rate -> Hyper-parameter

In [ ]:
# Training the model using the training loop

model = LinearRegressionModel()

loss_fn = nn.L1Loss()  # Mean Absolute Error (MAE)
# Experiment with nn.MSELoss()

optimizer = optim.SGD(params=model.parameters(), lr=0.0001)  # lr -> Learning rate -> Hyper-parameter
# Experiment with optim.ADAM()

torch.manual_seed(42)

epochs = 75  # A kind of a Hyper-parameter

train_loss_values = []
test_loss_values = []
epoch_count = []

for epoch in range(epochs):
    #print(f"Epoch {epoch+1}:")
    # Train the model
    model.train()

    # Forward pass (perform predictions based on current weight and bias)
    y_pred = model(X_train)

    # Calculate the loss
    loss = loss_fn(y_pred, y_train)
    #print(f"    Loss: {loss}")

    # Zero grad the optimizer
    optimizer.zero_grad()

    # Loss backwards (backwards pass)
    loss.backward()

    # Progress the optimizer -> updates the weight and bias - based on the loss gradients
    optimizer.step()

    # Evaluate the model
    model.eval()

    # Test the model
    with torch.inference_mode():
        test_pred = model(X_test)

        test_loss = loss_fn(test_pred, y_test.type(torch.float))

        if epoch % 10 == 0:
            epoch_count.append(epoch)
            train_loss_values.append(loss.detach().numpy())
            test_loss_values.append(test_loss.detach().numpy())
            print(f"Epoch: {epoch} | MAE Train Loss: {loss} | MAE Test Loss: {test_loss} ")



In [ ]:
model.state_dict()

In [ ]:
plt.plot(epoch_count, train_loss_values, label="Train Loss")
plt.plot(epoch_count, test_loss_values, label="Test Loss")
plt.title("Training and test loss curves")
plt.ylabel("Loss")
plt.xlabel("Epochs")
plt.legend()
plt.show()

#### L1Loss - Mean Absolute Error (MAE)

$$MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

- n: Total number of observations (iteration count)
- $y_i$: Actual value for i observation (actual value)
- $\hat{y}_i$: Predicted value for the i observation (predicted value)
- $|y_i - \hat{y}_i|$: Absolute difference (error) between the actual and predicted values

---

#### L2Loss - Mean-Squared-Loss (MSE)

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

---


##### Choosing between L1Loss and L2Loss:
- For training data with minimal noise / deviation (no outliers), MSE Loss (L2 Loss) is preferred.
- MSE leads to faster convergence (as errors are amplified)

- For training data with some noise / outliers, MAE Loss is preferred.
- As MSE also amplifies influence of outliers, can become inefficient in such scenarios
- MAE treats all error equally, influence of outliers would be minimal.

In summary, use MSE if your training data is clean, MAE otherwise.

---

##### Choosing Optimizers
  - SGD (Stochastic Gradient Descent) is generally preferred for very large training datasets with simpler model features. It has a fixed learning-rate and work well with Simple Neural Networks.
  - Adam (Adaptive Momentum Estimation) is generally preferred for complex model features. It's adaptive learning rate and bias correction can lead to faster convergence - but it is computation intensive than SGD. It is suitable for Deep Neural Networks.


### **Machine Learning Task Taxonomy**



| **Category**                         | **Sub-Category**                         | **Examples / Notes**                                    |
| ------------------------------------ | ---------------------------------------- | ------------------------------------------------------- |
| **Supervised Learning**              | **Classification**                       | Spam detection, disease diagnosis, image labeling       |
|                                      | **Regression**                           | Predicting prices, temperature, stock values            |
|                                      | **Ranking**                              | Search result ordering, recommender ranking             |
|                                      | **Forecasting / Time-Series Prediction** | Sales forecasting, weather prediction                   |
|                                      | **Anomaly Detection (Supervised)**       | Fraud detection with labeled anomalies                  |
| **Unsupervised Learning**            | **Clustering**                           | Customer segmentation, gene expression grouping         |
|                                      | **Dimensionality Reduction**             | PCA (Principal Component Analysis), t-SNE (t-distributed Stochastic Neighbour Embedding), UMAP (Uniform Manifold Approximation and Projection) for visualization or preprocessing     |
|                                      | **Anomaly Detection (Unsupervised)**     | Detecting outliers without labels                       |
|                                      | **Density Estimation**                   | Modeling probability distributions (e.g., GMM)          |
| **Semi- & Self-Supervised Learning** | **Semi-Supervised**                      | Few labeled + many unlabeled examples (pseudo-labeling) |
|                                      | **Self-Supervised**                      | Contrastive learning (SimCLR - Simple Framework For Contrastive Learning), masked modeling (BERT - Bidirectional Encoder Representations from Transformers)   |
| **Reinforcement Learning**           | **Classic RL**                           | Agents learning via reward feedback (Atari, robotics)   |
|                                      | **Inverse RL**                           | Inferring reward functions from expert demonstrations   |
| **Generative Modeling**              | **GANs** (Generative Adversarial Networks)                                | Image generation, super-resolution                      |
|                                      | **VAEs (Variational Autoencoders)**      | Latent-space learning for reconstruction                |
|                                      | **Diffusion Models**                     | Text-to-image (e.g., Stable Diffusion)                  |
|                                      | **Text-to-X / Image-to-X Translation**   | Style transfer, image captioning                        |
|                                      | **Language Modeling**                    | GPT, LLaMA — next-token prediction                      |
| **Multi-Task & Transfer Learning**   | **Transfer Learning**                    | Fine-tuning pretrained models (BERT, ResNet)            |
|                                      | **Multi-Task Learning**                  | Shared representations across related tasks             |
|                                      | **Meta-Learning**                        | Learning to adapt rapidly (MAML, Reptile)               |
| **Specialized / Applied Domains**    | **Computer Vision**                      | Object detection, segmentation, depth estimation        |
|                                      | **Natural Language Processing (NLP)**    | Sentiment analysis, summarization, translation          |
|                                      | **Speech & Audio**                       | Speech recognition, speaker identification              |
|                                      | **Recommender Systems**                  | Collaborative filtering, contextual bandits             |
| **Emerging / Hybrid Paradigms**      | **Causal Inference**                     | Understanding cause–effect relationships                |
|                                      | **Federated Learning**                   | Distributed training without sharing data               |
|                                      | **Active Learning**                      | Querying most informative samples for labeling          |
|                                      | **Continual / Lifelong Learning**        | Avoiding catastrophic forgetting                        |
|                                      | **Explainable AI (XAI)**                 | Model interpretability (LIME, SHAP)                     |
|                                      | **Graph Learning (GNNs)**                | Learning on networks: social, molecular, etc.           |

---



### An overview on Classification in ML

Classification task is one of the simplest forms of Machine Learning objectives. 
Classification teaches a computer to categorize things into categories. The model learns by looking at training data with labels (like emails marked "spam" or "not spam"). After learning, the model can decide which category new items belong to, like identifying if a new email is spam or not. For example, a classifier model can be trained on dataset of images labeled as either dogs or cats and it can be used to predict the class of new and unseen images as dogs or cats based on their features such as color, texture and shape.

![image](./classification_vs_regression.png)


##### Types of classification
 1. Binary classification: Sorts items into two distinct categories. (e.g. spam vs not spam)
 2. Multiclass classification: Sorts items into more than two categories. (e.g., plane, car, bird, animal)
   
![image](./binary_vs_multiclass.png)

##### The loss (criterion) functions

While Regression models use loss functions like L1Loss (MAE) or L2Loss (MSE), Classification models use one of the following loss functions:
- **Cross-Entropy Loss**: This is a widely used loss function for classification, especially when the model's output is a probability distribution over classes. It measures the difference between the predicted probability distribution and the true distribution (e.g., one-hot encoded labels). There are variations like Binary Cross-Entropy for binary classification and Categorical Cross-Entropy for multi-class classification.

- **Log Loss**: This is another name for Cross-Entropy Loss, and it's used in binary classification problems.  (also known as BCELoss)

- **Hinge Loss**: This loss function is particularly suited for Support Vector Machines (SVMs) and is used to maximize the margin between different classes. It penalizes misclassified samples and samples within the margin. 

- **Squared Hinge Loss**: Similar to Hinge Loss, but it squares the distance from the margin, resulting in a stronger penalty for misclassified samples. 


- **Zero-One Loss**: This is a simple loss function that assigns a loss of 1 for incorrect classifications and 0 for correct classifications. 


#### Activation Functions in the hidden layers

![image](./activation_functions.webp)

##### Preparing our environment

In [ ]:
# conda create -n apaimlt scikit-learn pandas matplotlib pytorch torchvision torchaudio ipywidgets tqdm -y

In [ ]:
import torch
torch.__version__

#### Building a Feed-Forward Neural Network For Classification

FFNN (also known as a MultiLayer Perceptron or MLP) is a neural network with a simple architecture. 
It features an input layer - made up of neurons representing the number of features, one or more hidden (fully connected) layers - made of neurons that gradually converge - and an output layer made up of neurons matching the output labels.


![image](./ffnn.jpg)

In [ ]:
28 * 28


In this example, we will build a classification model for the MNIST (*Modified National Institute of Standards and Technology*) dataset, which is made up of handwritten digits from 0 to 9 in black-and-white 28x28 pixel images

![image](./mlp-mnist.png)

We'll process the dataset, build our model, and then train our model. Afterwards, we'll do a short dive into what the model has actually learned.

##### Data Processing

Let's start by importing all the modules we'll need. The main ones we need to import are:
- torch for general PyTorch functionality
- torch.nn and torch.nn.functional for neural network based functions
- torch.optim for our optimizer which will update the parameters of our neural network
- torch.utils.data for handling the dataset
- torchvision.transforms for data augmentation
- torchvision.datasets for loading the dataset
- sklearn's metrics for visualizing a confusion matrix
- sklearn's decomposition and manifold for visualizing the neural network's representations in two dimensions
- matplotlib for plotting

In [ ]:
# conda install tqdm -y

In [ ]:
# Basic imports for building a neural network with PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data


In [ ]:
# Additional imports for data handling and visualization
import torchvision.transforms as transforms
import torchvision.datasets as datasets


In [ ]:
# Imports for model evaluation and visualization
from sklearn import metrics
from sklearn import decomposition
from sklearn import manifold
from tqdm.notebook import trange, tqdm
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
# Additional utility imports
import copy
import random
import time

To ensure we get reproducible results we set the random seed for Python, Numpy and PyTorch.

In [ ]:
SEED = 42

random.seed(SEED) # Core Python RNG
np.random.seed(SEED) # NumPy RNG
torch.manual_seed(SEED) # PyTorch RNG
torch.mps.manual_seed(SEED) # PyTorch MPS RNG
torch.backends.mps.deterministic = True

The first thing we'll do is load our dataset.

This will automatically download the training set for the MNIST dataset and save it in a folder called `.data`. It will create the folder if it does not exist.

In [ ]:
datasets.MNIST?

In [ ]:
ROOT = '../data'

train_data = datasets.MNIST(root=ROOT,
                            train=True,
                            download=True)

In [ ]:
i = 1234
print(train_data[i][1])
plt.imshow(train_data[i][0], cmap='gray')

In [ ]:
plt.imshow(train_data.data[2], cmap='gray')

In [ ]:
train_data.data[0].ravel().unique()

In [ ]:
train_data.data.shape

In [ ]:
train_data.data.float().std()

Next, we want to *normalize* our data. This means we want it to have a mean of zero and a standard deviation of one. 

Why do we want to do this? Normalizing our data allows our models to train faster and to also help them avoid local minima, i.e. train more reliably.

We normalize our data by subtracting the mean and dividing by the standard deviation of our dataset. First, we need to calculate the mean and standard deviation. **Note**: it is important that the mean and standard deviation are only calculated over the training set and not the test set. We do not want to use any information at all from the test set and should only look at it when we are calculating our test loss.

To calculate the means and standard deviations we get the actual data (the images) using the `.data.` attribute of our training data, convert them into floating point numbers, and then use the built-in `mean` and `std` functions to calculate the mean and standard deviation, respectively. The image data has values between 0-255, which we want to scale between 0-1, so we divide by 255.

In [ ]:
mean = train_data.data.float().mean() / 255
std = train_data.data.float().std() / 255

In [ ]:
train_data.data[0].ravel().unique()

In [ ]:
mean, std

In [ ]:
t = train_data.data.clone()
t = (t - mean) / std
t[0].ravel().unique()

In [ ]:
print(f'Calculated mean: {mean}')
print(f'Calculated std: {std}')

Now we've calculated our mean and standard deviation how do we actually use them? We use Torchvision's `transforms`. 

A `transform` states how our data should be augmented and processed. Data augmentation involves manipulating the available training data in a way that artificially creates more training examples. We use `transforms.Compose` to built a list of transformations that will be applied to the image. 

The transforms we use are:
- `RandomRotation` - randomly rotates the image between `(-x, +x)` degrees, where we have set `x = 5`. Note, the `fill=(0,)` is due to a [bug](https://github.com/pytorch/vision/issues/1759) in some versions of torchvision. 
- `RandomCrop` - this first adds `padding` around our image, 2 pixels here, to artificially make it bigger, before taking a random `28x28` square crop of the image.
- `ToTensor()` - this converts the image from a PIL image into a PyTorch tensor.
- `Normalize` - this subtracts the mean and divides by the standard deviations given. 

The first two transformations have to be applied before `ToTensor` as they should both be applied on a PIL image. `Normalize` should only be applied to the images after they have been converted into a tensor. See the Torchvision documentation for [transforms that should be applied to PIL images](https://pytorch.org/vision/stable/transforms.html#transforms-on-pil-image-only) and [transforms that should be applied on tensors](https://pytorch.org/vision/stable/transforms.html#transforms-on-torch-tensor-only).

We have two lists of transforms, a train and a test transform. The train transforms are to artificially create more examples for our model to train on. We do not augment our test data in the same way, as we want a consistent set of examples to evaluate our final model on. The test data, however, should still be normalized.

In [ ]:
train_data.data[0].shape

In [ ]:
perspective_transformer = transforms.RandomPerspective(distortion_scale=0.6, p=1.0)
perspective_imgs = perspective_transformer(train_data[0][0])
perspective_imgs

In [ ]:
perspective_transformer = transforms.RandomPerspective(distortion_scale=0.6, p=1.0)
perspective_imgs = [perspective_transformer(train_data[0][0]) for _ in range(10)]
orig_img = train_data[0][0]
plt.imshow(orig_img, cmap='gray')
plt.show()
for i, img in enumerate(perspective_imgs):
    plt.subplot(1, 10, i+1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')

In [ ]:
perspective_transformer = transforms.RandomRotation(degrees=30)
perspective_imgs = [perspective_transformer(train_data[0][0]) for _ in range(10)]
orig_img = train_data[0][0]
plt.imshow(orig_img, cmap='gray')
plt.show()
for i, img in enumerate(perspective_imgs):
    plt.subplot(1, 10, i+1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')

In [ ]:
perspective_transformer = transforms.RandomCrop(size=(20, 20))
perspective_imgs = [perspective_transformer(train_data[0][0]) for _ in range(10)]
orig_img = train_data[0][0]
plt.imshow(orig_img, cmap='gray')
plt.show()
for i, img in enumerate(perspective_imgs):
    plt.subplot(1, 10, i+1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')

In [ ]:
train_transforms = transforms.Compose([
                            transforms.RandomPerspective(distortion_scale=0.6, p=1.0),
                            transforms.RandomRotation(5, fill=(0,)),
                            transforms.RandomCrop(28, padding=2),
                            transforms.ToTensor(),
                            transforms.Normalize(mean=[mean], std=[std])
                                      ])

test_transforms = transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize(mean=[mean], std=[std])
                                     ])

print(train_transforms)
print(test_transforms)

Now we have defined our transforms we can then load the train and test data with the relevant transforms defined.

In [ ]:
train_data = datasets.MNIST(root=ROOT,
                            train=True,
                            download=True,
                            transform=train_transforms
)

test_data = datasets.MNIST(root=ROOT,
                           train=False,
                           download=True,
                           transform=test_transforms
)

In [ ]:
plt.imshow(train_data.data[0], cmap="gray")

We can simply check the `len` of the datasets to see how many examples are within each.

In [ ]:
print(f'Number of training examples: {len(train_data)}')
print(f'Number of testing examples: {len(test_data)}')

We can get a look at some of the images within our dataset to see what we're working with. The function below plots a square grid of images. If you supply less than a complete square number of images it will ignore the last few.

In [ ]:
def plot_images(images):

    n_images = len(images)

    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))

    fig = plt.figure()
    for i in range(rows*cols):
        ax = fig.add_subplot(rows, cols, i+1)
        ax.imshow(images[i].view(28, 28).cpu().numpy(), cmap='bone')
        ax.axis('off')

Let's load 25 images. These will have been processed through our transforms, so will be randomly rotated and cropped.

It's a good practice to see your data with your transforms applied, so you can ensure they look sensible. For example, it wouldn't make sense to flip the digits horizontally or vertically unless you are expecting to see what in your test data.

In [ ]:
train_data[0][0].shape

In [ ]:
N_IMAGES = 64

images = [image for image, label in [train_data[i] for i in range(N_IMAGES)]]

plot_images(images)

In [ ]:
train_data.data.shape

The MNIST dataset comes with a training and test set, but not a validation set. We want to use a validation set to check how well our model performs on unseen data. Why don't we just use the test data? We should only be measuring our performance over the test set once, after all training is done. We can think of the validation set as a proxy test set we are allowed to look at as much as we want. 

Furthermore, we create a validation set, taking 10% of the training set. **Note:** the validation set should always be created from the training set. Never take the validation set from the test set. When researchers publish research papers they should be comparing performance across the test set and the only way to ensure this is a fair comparison is for all researchers to use the same test set. If the validation set is taken from the test set, then the test set is not the same as everyone else's and the results cannot be compared against each other.

First, we have to define the exact number of examples that we want to be in each split of the training/validation sets.

In [ ]:
VALID_RATIO = 0.9

n_train_examples = int(len(train_data) * VALID_RATIO)
n_valid_examples = len(train_data) - n_train_examples
n_train_examples, n_valid_examples

Then, we use the `random_split` function to take a random 10% of the training set to use as a validation set. The remaining 90% will stay as the training set.

In [ ]:
train_data, valid_data = data.random_split(train_data,
                                           [n_train_examples, n_valid_examples])

We can print out the number of examples again to check our splits are correct.

In [ ]:
print(f'Number of training examples: {len(train_data)}')
print(f'Number of validation examples: {len(valid_data)}')
print(f'Number of testing examples: {len(test_data)}')

One thing to consider is that as the validation set has been created from the training set it has the same transforms as the training set, with the random rotating and cropping. As we want our validation set to act as a proxy for the test set, it should also be fixed, without any random augmentation. 

First, let's see what 25 of the images within the validation set look like with the training transforms:

In [ ]:
N_IMAGES = 25

images = [image for image, label in [valid_data[i] for i in range(N_IMAGES)]]

plot_images(images)

We can now simply replace the validation set's transform by overwriting it with our test transforms from above.

As the validation set is a `Subset` of the training set, if we change the transforms of one, then by default Torchvision will change the transforms of the other. To stop this from happening, we make a `deepcopy` of the validation data.

In [ ]:
valid_data = copy.deepcopy(valid_data)
valid_data.dataset.transform = test_transforms

To double check we've correctly replaced the training transforms, we can view the same set of images and notice how they're more central (no random cropping) and have a more standard orientation (no random rotations).

In [ ]:
N_IMAGES = 25

images = [image for image, label in [valid_data[i] for i in range(N_IMAGES)]]

plot_images(images)

Next, we'll define a `DataLoader` for each of the training/validation/test sets. We can iterate over these, and they will yield batches of images and labels which we can use to train our model.

We only need to shuffle our training set as it will be used for stochastic gradient descent, and we want each batch to be different between epochs. As we aren't using the validation or test sets to update our model parameters, they do not need to be shuffled.

Ideally, we want to use the biggest batch size that we can. The 64 here is relatively small and can be increased if our hardware can handle it.

In [ ]:
BATCH_SIZE = 64

train_iterator = data.DataLoader(train_data,
                                 shuffle=True,
                                 batch_size=BATCH_SIZE)

valid_iterator = data.DataLoader(valid_data,
                                 batch_size=BATCH_SIZE)

test_iterator = data.DataLoader(test_data,
                                batch_size=BATCH_SIZE)

### Defining the Model

Our model will be a neural network, specifically a multilayer perceptron (MLP) with two hidden layers. The image below shows the archicture of the model. 

![](https://github.com/bentrevett/pytorch-image-classification/blob/master/assets/mlp-mnist.png?raw=1)

Specifically, first we will flatten our 1x28x28 (1 color channel, 28 pixels height and width) image into a 784 element vector, also called 784 *features*. We flatten our input, as MLPs cannot handle two or three-dimensional data. Next, the 784 dimensional input is passed through the first hidden layer to transform it into 250 dimensions. Then, another hidden layer, which will transform it to 100 dimensions. Finally, an output layer which will transform it into a 10 dimensional vector. The output dimension should equal the number of classes within your data. Here we have ten digits, 0 - 9, so need our output to be 10 dimensions.

The transformation between 784 to 250, 250 to 100 and 100 to 10 dimensions are done by `Linear` layers. These are also known as fully connected or affine layers. In these layers, every element in one layer is connected to every element in the next. We can think of these elements as *neurons*, as this architecture is inspired by how the human brain is made of millions of interconnected nodes, also called neurons. 

Each connection between a neuron in one layer and a neuron in the next has a *weight* associated with it. The input to one neuron is the sum of the weighted values of all neurons in the previous layer connected to it, plus a weighted bias term, where the bias value is always 1. The neuron then applies an *activation function* to this weighted sum. This activation function is a non-linear function that allows the neural network to learn non-linear functions between inputs and outputs. 

We define our MLP below, which consists of three linear layers. We first take the input batch of images and flatten them, so they can be passed into the linear layers. We then pass them through the first linear layer, `input_fc`, which calculates the weighted sum of the inputs, and then apply the *ReLU* (rectified linear unit) activation function elementwise. This result is then passed through another linear layer, `hidden_fc`, again applying the same activation function elementwise. Finally, we pass this through the final linear layer, `output_fc`. We return not only the output but also the second hidden layer as we will do some analysis on it later.

The ReLU activation function is a popular non-linear function that is simply $max(0, x)$, where $x$ is the weighted sum of the inputs to that neuron. Other activation functions used are hyperbolic tan (tanh) and sigmoid function, however ReLU is the most commonly used.

![](https://github.com/bentrevett/pytorch-image-classification/blob/master/assets/relu.png?raw=1)


One thing to note is that we do not use an activation function on the input directly or on the output. You should never use activation functions directly on the input, i.e. `F.relu(x)`. PyTorch combines activation functions to be applied on the output with the functions which calculate the *loss*, also known as *error* or *cost*, of a neural network. This is done for numerical stability.

Why did we choose hidden dimensions of 250 and 100 elements? Why did we only have two hidden layers? There is no magic formula to tell us how many layers to use and how many neurons to have in each layer, and there is most probably a better set of values. However, the general idea is that neural networks extract features from data. Layers closer to the input learn to extract general features (e.g. lines, curves, edges), whilst later layers combine the features extracted from the previous layer into more high level features (e.g. the intersection of two lines making a cross, multiple curves make a circle). We force our neural network to learn these features by reducing the number of neurons in each layer. This way, it has to learn to compress information by extracting only the useful and general features. Thus, we want a neural network with multiple layers and some sort of information compression (reduced number of neurons in subsequent layers).

In [ ]:
class MLP(nn.Module): # Multi-Layered-Perceptron a.k.a Feed-Forward-Network
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.input_layer = nn.Linear(input_dim, 350)
        self.fc1 = nn.Linear(350, 170)
        self.fc2 = nn.Linear(170, 100)
        self.output_layer = nn.Linear(100, output_dim)

    def forward(self, x):

        # x = [batch size, height, width]

        batch_size = x.shape[0]

        x = x.view(batch_size, -1)

        x = F.relu(self.input_layer(x))

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # h_2 = [batch size, 100]

        y_pred = self.output_layer(x)

        # y_pred = [batch size, output dim]

        return y_pred, x

We'll define our model by creating an instance of it and setting the correct input and output dimensions.

In [ ]:
INPUT_DIM = 28 * 28
OUTPUT_DIM = 10

model = MLP(INPUT_DIM, OUTPUT_DIM)

We can also create a small function to calculate the number of trainable parameters (weights and biases) in our model - in case all of our parameters are trainable.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

The first layer has 784 neurons connected to 250 neurons, so 784*250 weighted connections plus 250 bias terms.

The second layer has 250 neurons connected to 100 neurons, 250*100 weighted connections plus 100 bias terms.

The third layer has 100 neurons connected to 10 neurons, 100*10 weighted connections plus 10 bias terms.

$$784 \cdot 250 + 250 + 250 \cdot 100 + 100 + 100 \cdot 10 + 10 = 222,360 $$

In [ ]:
print(f'The model has {count_parameters(model):,} trainable parameters')

### Training the Model

Next, we'll define our optimizer. This is the algorithm we will use to update the parameters of our model with respect to the loss calculated on the data.

We aren't going to go into too much detail on how neural networks are trained (see [this](http://neuralnetworksanddeeplearning.com/) article if you want to know how) but the gist is:
- pass a batch of data through your model
- calculate the loss of your batch by comparing your model's predictions against the actual labels
- calculate the gradient of each of your parameters with respect to the loss
- update each of your parameters by subtracting their gradient multiplied by a small *learning rate* parameter

We use the *Adam* algorithm with the default parameters to update our model. Improved results could be obtained by searching over different optimizers and learning rates, however default Adam is usually a good starting off point. Check out [this](https://ruder.io/optimizing-gradient-descent/) article if you want to learn more about the different optimization algorithms commonly used for neural networks.

In [ ]:
optimizer = optim.Adam(model.parameters())

Then, we define a *criterion*, PyTorch's name for a loss/cost/error function. This function will take in your model's predictions with the actual labels and then compute the loss/cost/error of your model with its current parameters.

`CrossEntropyLoss` both computes the *softmax* activation function on the supplied predictions as well as the actual loss via *negative log likelihood*. 

Briefly, the softmax function is:
  ![Softmax Formula](./softmax_formula.png)

This turns out 10 dimensional output, where each element is an unbounded real number, into a probability distribution over 10 elements. That is, all values are between 0 and 1, and together they all sum to 1. 

Why do we turn things into a probability distribution? So we can use negative log likelihood for our loss function, as it expects probabilities. PyTorch calculates negative log likelihood for a single example via:

$$\text{negative log likelihood }(\mathbf{\hat{y}}, y) = -\log \big( \text{softmax}(\mathbf{\hat{y}})[y] \big)$$

$\mathbf{\hat{y}}$ is the $\mathbb{R}^{10}$ output, from our neural network, whereas $y$ is the label, an integer representing the class. The loss is the negative log of the class index of the softmax. For example:

$$\mathbf{\hat{y}} = [5,1,1,1,1,1,1,1,1,1]$$

$$\text{softmax }(\mathbf{\hat{y}}) = [0.8585, 0.0157, 0.0157, 0.0157, 0.0157, 0.0157, 0.0157, 0.0157, 0.0157, 0.0157]$$

If the label was class zero, the loss would be:

$$\text{negative log likelihood }(\mathbf{\hat{y}}, 0) = - \log(0.8585) = 0.153 \dots$$

If the label was class five, the loss would be:

$$\text{negative log likelihood }(\mathbf{\hat{y}}, 5) = - \log(0.0157) = 4.154 \dots$$

So, intuitively, as your model's output corresponding to the correct class index increases, your loss decreases.

In [ ]:
def softmax(x):
    """
    Compute softmax values for each element in x.
    Args:
        x (array-like): Input vector or matrix.
    Returns:
        numpy.ndarray: Softmax probabilities.
    """
    import torch
    x = torch.tensor(x)
    # Subtract max(x) for numerical stability
    e_x = torch.exp(x - torch.max(x))
    return e_x / torch.sum(e_x)

scores = [2.0, 1.0, 0.1]
probabilities = softmax(scores)
print("Softmax probabilities:", probabilities)
print("Sum:", torch.sum(probabilities))


import torch.nn.functional as F
print(F.softmax(torch.tensor([2.0, 1.0, 0.1]), dim=0))

In [ ]:
import torch

def softmax(x):
    x = torch.tensor(x)
    e_x = torch.exp(x - torch.max(x))  # for numerical stability
    return e_x / torch.sum(e_x)

def cross_entropy_loss(y_pred, y_true):
    """
    Compute the Cross-Entropy Loss.
    Args:
        y_pred (array-like): Raw model outputs (logits) or probabilities.
        y_true (array-like): One-hot encoded true labels or class indices.
    Returns:
        float: Cross entropy loss value.
    """
    y_pred = torch.tensor(y_pred)
    
    # If logits are provided, apply softmax
    if y_pred.ndim == 1 or torch.any(y_pred < 0) or torch.any(y_pred > 1):
        y_pred = softmax(y_pred)
    
    # Convert y_true to one-hot if needed
    if y_true.ndim == 0 or torch.issubdtype(type(y_true), torch.int64):
        y_true = torch.eye(len(y_pred))[y_true]
    
    # Clip predictions to avoid log(0)
    y_pred = torch.clamp(y_pred, 1e-12, 1.0)
    
    # Compute cross-entropy
    loss = -torch.sum(y_true * torch.log(y_pred))
    return loss

logits = [2.0, 1.0, 0.1] # 3 classes as logits (raw model outputs)
true_class = torch.tensor(0)  # The correct class index

loss = cross_entropy_loss(logits, true_class)
print("Cross Entropy Loss:", loss)



In [ ]:
criterion = nn.CrossEntropyLoss()

We then define `device`. This is used to place your model and data on to a GPU, if you have one.

In [ ]:
device = torch.accelerator.current_accelerator().type \
    if torch.accelerator.is_available else 'cpu'


We place our model and criterion on to the device by using the `.to` method.

In [ ]:
model = model.to(device)
criterion = criterion.to(device)
print(model, criterion)

Next, we'll define a function to calculate the accuracy of our model. This takes the index of the highest value for your prediction and compares it against the actual class label. We then divide how many our model got correct by the amount in the batch to calculate accuracy across the batch.

In [ ]:
def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc

We finally define our training loop.

This will:
- put our model into `train` mode
- iterate over our dataloader, returning batches of (image, label)
- place the batch on to our GPU, if we have one
- clear the gradients calculated from the last batch
- pass our batch of images, `x`, through to model to get predictions, `y_pred`
- calculate the loss between our predictions and the actual labels
- calculate the accuracy between our predictions and the actual labels
- calculate the gradients of each parameter
- update the parameters by taking an optimizer step
- update our metrics

Some layers act differently when training and evaluating the model that contains them, hence why we must tell our model we are in "training" mode. The model we are using here does not use any of those layers, however it is good practice to get used to putting your model in training mode.

In [ ]:
def train(model, iterator, optimizer, criterion, device):

    epoch_loss = 0
    epoch_acc = 0

    model.train()

    for (x, y) in tqdm(iterator, desc="Training", leave=False):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred, _ = model(x)

        loss = criterion(y_pred, y)

        acc = calculate_accuracy(y_pred, y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

The evaluation loop is similar to the training loop. The differences are:
- we put our model into evaluation mode with `model.eval()`
- we wrap the iterations inside a `with torch.no_grad()`
- we do not zero gradients as we are not calculating any
- we do not calculate gradients as we are not updating parameters
- we do not take an optimizer step as we are not calculating gradients

`torch.no_grad()` ensures that gradients are not calculated for whatever is inside the `with` block. As our model will not have to calculate gradients, it will be faster and use less memory. 

In [ ]:
def evaluate(model, iterator, criterion, device):

    epoch_loss = 0
    epoch_acc = 0

    model.eval()

    with torch.no_grad():

        for (x, y) in tqdm(iterator, desc="Evaluating", leave=False):

            x = x.to(device)
            y = y.to(device)

            y_pred, _ = model(x)

            loss = criterion(y_pred, y)

            acc = calculate_accuracy(y_pred, y)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

The final step before training is to define a small function to tell us how long an epoch took.

In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

We're finally ready to train!

During each epoch we calculate the training loss and accuracy, followed by the validation loss and accuracy. We then check if the validation loss achieved is the best validation loss we have seen. If so, we save our model's parameters (called a `state_dict`).

In [ ]:
pip install ipywidgets

In [ ]:
EPOCHS = 10

best_valid_loss = float('inf')

for epoch in trange(EPOCHS):

    start_time = time.monotonic()

    train_loss, train_acc = train(model, train_iterator, optimizer, criterion, device)
    valid_loss, valid_acc = evaluate(model, valid_iterator, criterion, device)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'model1.pt')

    end_time = time.monotonic()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

Afterwards, we load our the parameters of the model that achieved the best validation loss and then use this to evaluate our model on the test set.

In [ ]:
model.load_state_dict(torch.load('model1.pt', weights_only=True))

test_loss, test_acc = evaluate(model, test_iterator, criterion, device)

Our model achieves around 95% to 97% accuracy on the test set.

This can be improved by tweaking hyperparameters, e.g. number of layers, number of neurons per layer, optimization algorithm used, learning rate, etc. 

In [ ]:
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

### Examining the Model

Now we've trained our model, there are a few things we can look at. Most of these are simple exploratory analysis, but they can offer some insights into your model.

An important thing to do is check what examples your model gets wrong and ensure that they're reasonable mistakes.

The function below will return the model's predictions over a given dataset. It will return the inputs (image) the outputs (model predictions) and the ground truth labels.

In [ ]:
def get_predictions(model, iterator, device):

    model.eval()

    images = []
    labels = []
    probs = []

    with torch.no_grad():

        for (x, y) in iterator:

            x = x.to(device)

            y_pred, _ = model(x)

            y_prob = F.softmax(y_pred, dim=-1)

            images.append(x.cpu())
            labels.append(y.cpu())
            probs.append(y_prob.cpu())

    images = torch.cat(images, dim=0)
    labels = torch.cat(labels, dim=0)
    probs = torch.cat(probs, dim=0)

    return images, labels, probs

We can then get these predictions and, by taking the index of the highest predicted probability, get the predicted labels.

In [ ]:
images, labels, probs = get_predictions(model, test_iterator, device)

pred_labels = torch.argmax(probs, 1)
print(pred_labels)

Then, we can make a confusion matrix from our actual labels and our predicted labels.

In [ ]:
def plot_confusion_matrix(labels, pred_labels):

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(1, 1, 1)
    cm = metrics.confusion_matrix(labels, pred_labels)
    cm = metrics.ConfusionMatrixDisplay(cm, display_labels=range(10))
    cm.plot(values_format='d', cmap='Blues', ax=ax)

The results seem reasonable enough, the most confused predictions-actuals are: 3-5 and 2-7.

In [ ]:
plot_confusion_matrix(labels, pred_labels)

Next, for each of our examples, we can check if our predicted label matches our actual label.

In [ ]:
corrects = torch.eq(labels, pred_labels)

We can then loop through all of the examples over our model's predictions and store all the examples the model got incorrect into an array.

Then, we sort these incorrect examples by how confident they were, with the most confident being first.

In [ ]:
incorrect_examples = []

for image, label, prob, correct in zip(images, labels, probs, corrects):
    if not correct:
        incorrect_examples.append((image, label, prob))

incorrect_examples.sort(reverse=True,
                        key=lambda x: torch.max(x[2], dim=0).values)

We can then plot the incorrectly predicted images along with how confident they were on the actual label and how confident they were at the incorrect label.

In [ ]:
def plot_most_incorrect(incorrect, n_images):

    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))

    fig = plt.figure(figsize=(20, 10))
    for i in range(rows*cols):
        ax = fig.add_subplot(rows, cols, i+1)
        image, true_label, probs = incorrect[i]
        true_prob = probs[true_label]
        incorrect_prob, incorrect_label = torch.max(probs, dim=0)
        ax.imshow(image.view(28, 28).cpu().numpy(), cmap='bone')
        ax.set_title(f'true label: {true_label} ({true_prob:.3f})\n'
                     f'pred label: {incorrect_label} ({incorrect_prob:.3f})')
        ax.axis('off')
    fig.subplots_adjust(hspace=0.5)

Below we can see the 25 images the model got incorrect and was most confident about.

A lot of these digits are irregular, so it's difficult for the model to do well on these. The images that do look fine, if you squint you can sort of see why the model got it wrong.

Why is the neural network so confident on the irregular digits? Surely if it's a weird looking digit then the output of the softmax should be close to evenly distributed across a few digits the model isn't sure about, right? Well, no. The model has been trained to only be incredibly confident about its predictions, and thus when it sees an image it will always be confident about what it is.

In [ ]:
N_IMAGES = 25

plot_most_incorrect(incorrect_examples, N_IMAGES)

Another thing we can do is get the output and intermediate representations from the model and try to visualize them.

The function below loops through the provided dataset and gets the output from the model and the intermediate representation from the layer before that, the second hidden layer.

In [ ]:
def get_representations(model, iterator, device):

    model.eval()

    outputs = []
    intermediates = []
    labels = []

    with torch.no_grad():

        for (x, y) in tqdm(iterator):

            x = x.to(device)

            y_pred, h = model(x)

            outputs.append(y_pred.cpu())
            intermediates.append(h.cpu())
            labels.append(y)

    outputs = torch.cat(outputs, dim=0)
    intermediates = torch.cat(intermediates, dim=0)
    labels = torch.cat(labels, dim=0)

    return outputs, intermediates, labels

We run the function to get the representations.

In [ ]:
outputs, intermediates, labels = get_representations(model,
                                                     train_iterator,
                                                     device)

The data we want to visualize is in ten dimensions and 100 dimensions. We want to get this down to two dimensions, so we can actually plot it. 

The first technique we'll use is PCA (principal component analysis). First, we'll define a function to calculate the PCA of our data, and then we'll define a function to plot it.

In [ ]:
def get_pca(data, n_components=2):
    pca = decomposition.PCA()
    pca.n_components = n_components
    pca_data = pca.fit_transform(data)
    return pca_data

In [ ]:
def plot_representations(data, labels, n_images=None):
    if n_images is not None:
        data = data[:n_images]
        labels = labels[:n_images]
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)
    scatter = ax.scatter(data[:, 0], data[:, 1], c=labels, cmap='tab10')
    handles, labels = scatter.legend_elements()
    ax.legend(handles=handles, labels=labels)

First, we plot the representations from the ten dimensional output layer, reduced down to two dimensions.

In [ ]:
output_pca_data = get_pca(outputs)
plot_representations(output_pca_data, labels)

Next, we'll plot the outputs of the second hidden layer. 

The clusters seem similar to the one above. In fact if we rotated the below image anti-clockwise it wouldn't be too far off the PCA of the output representations.

In [ ]:
intermediate_pca_data = get_pca(intermediates)
plot_representations(intermediate_pca_data, labels)

An alternative to PCA is t-SNE (t-distributed stochastic neighbor embedding). 

This is commonly thought of as being "better" than PCA, although it can be [misinterpreted](https://distill.pub/2016/misread-tsne/).

In [ ]:
def get_tsne(data, n_components=2, n_images=None):
    if n_images is not None:
        data = data[:n_images]
    tsne = manifold.TSNE(n_components=n_components, random_state=0)
    tsne_data = tsne.fit_transform(data)
    return tsne_data

t-SNE is very slow, so we only compute it on a subset of the representations.

The classes look very well separated, and it is possible to use [k-NN](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm) on this representation to achieve decent accuracy.

In [ ]:
N_IMAGES = 5_000

output_tsne_data = get_tsne(outputs, n_images=N_IMAGES)
plot_representations(output_tsne_data, labels, n_images=N_IMAGES)

We plot the intermediate representations on the same subset.

Again, the classes look well separated, though less so than the output representations. This is because these representations are intermediate features that the neural network has extracted and will use them in further layers to weigh up the evidence of what digit is in the image. Hence, in theory, the classes should become more separated the closer we are to the output layer, which is exactly what we see here.

In [ ]:
intermediate_tsne_data = get_tsne(intermediates, n_images=N_IMAGES)
plot_representations(intermediate_tsne_data, labels, n_images=N_IMAGES)

Another experiment we can do is try and generate fake digits. 

The function below will repeatedly generate random noise and feed it through the model and find the most confidently generated digit for the desired class.

In [ ]:
def imagine_digit(model, digit, device, n_iterations=50_000):

    model.eval()

    best_prob = 0
    best_image = None

    with torch.no_grad():

        for _ in trange(n_iterations):

            x = torch.randn(32, 28, 28).to(device)

            y_pred, _ = model(x)

            preds = F.softmax(y_pred, dim=-1)

            _best_prob, index = torch.max(preds[:, digit], dim=0)

            if _best_prob > best_prob:
                best_prob = _best_prob
                best_image = x[index]

    return best_image, best_prob

Let's try and generate a perfect three.

In [ ]:
DIGIT = 3

best_image, best_prob = imagine_digit(model, DIGIT, device)

Looking at the best probability achieved, we have a digit that the model is 100% confident is a three. 

In [ ]:

print(f'Best image probability: {best_prob.item()*100:.2f}%')

Unfortunately, the imagined perfect three just looks like random noise.

As mentioned before, the model has only been trained to be incredibly confident with its predictions, so when faced with random noise will try and classify it as something. 

It is also possible that the model is *overfitting* on the training set - that there is a common pattern in handwritten threes within the training set, but it's not the pattern we want our model to learn.

In [ ]:
plt.imshow(best_image.cpu().numpy(), cmap='bone')
plt.axis('off');

Finally, we can plot the weights in the first layer of our model. 

The hope is that there's maybe one neuron in this first layer that's learned to look for certain patterns in the input and thus has high weight values indicating this pattern. If we then plot these weights, we should see these patterns.

In [ ]:
def plot_weights(weights, n_weights):

    rows = int(np.sqrt(n_weights))
    cols = int(np.sqrt(n_weights))

    fig = plt.figure(figsize=(20, 10))
    for i in range(rows*cols):
        ax = fig.add_subplot(rows, cols, i+1)
        ax.imshow(weights[i].view(28, 28).cpu().numpy(), cmap='bone')
        ax.axis('off')

Looking at these weights we see a few of them look like random noise but some of them do have weird patterns within them. These patterns show "ghostly" image looking shapes, but are clearly not images.

In [ ]:
N_WEIGHTS = 25

weights = model.input_layer.weight.data

plot_weights(weights, N_WEIGHTS)

### Next

- We will look into CNNs - LeNet and AlexNet
- Learn about FFNNs vs RNN vs CNN models
- Examples on training CIFAR image datasets